<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Getting-Started" data-toc-modified-id="Getting-Started-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Getting Started</a></span><ul class="toc-item"><li><span><a href="#Load-blank-exo-dictionary" data-toc-modified-id="Load-blank-exo-dictionary-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Load blank exo dictionary</a></span></li><li><span><a href="#Load-in-instrument-dictionary-(OPTIONAL)" data-toc-modified-id="Load-in-instrument-dictionary-(OPTIONAL)-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Load in instrument dictionary (OPTIONAL)</a></span></li><li><span><a href="#Running-PandExo" data-toc-modified-id="Running-PandExo-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Running PandExo</a></span></li></ul></li></ul></div>

# Comparing NIRISS SOSS Subarrays (incl new multistripe)

Before starting here, all the instructions on the [installation page](https://natashabatalha.github.io/PandExo/installation.html) should be completed! 

Here you will learn how to: 

- edit NIRISS SOSS subarrays 

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandexo.engine.justdoit as jdi # THIS IS THE HOLY GRAIL OF PANDEXO
import numpy as np
import os
import matplotlib.pyplot as plt

Make sure that your environment path is set to match the correct version of pandeia

In [ ]:
print(os.environ['pandeia_refdata'] )
import pandeia.engine
pandeia.engine.pandeia_version()

## Load blank exo dictionary

To start, load in a blank exoplanet dictionary with empty keys. You will fill these out for yourself in the next step. 

In [ ]:
exo_dict = jdi.load_exo_dict()

### Set Standard Exo Inputs 

In [ ]:
exo_dict['observation']['sat_level'] = 100    #saturation level in percent of full well 
exo_dict['observation']['sat_unit'] = '%'
exo_dict['observation']['noccultations'] = 1 #number of transits 
exo_dict['observation']['R'] = None          #fixed binning. I usually suggest ZERO binning.. you can always bin later 
                                             #without having to redo the calcualtion
exo_dict['observation']['baseline_unit'] = 'total'  #Defines how you specify out of transit observing time
                                                    #'frac' : fraction of time in transit versus out = in/out 
                                                    #'total' : total observing time (seconds)
exo_dict['observation']['baseline'] = 4.0*60.0*60.0 #in accordance with what was specified above (total observing time)

exo_dict['observation']['noise_floor'] = 0   #this can be a fixed level or it can be a filepath 
                                             #to a wavelength dependent noise floor solution (units are ppm)

#OPTION 1 get start from database
exo_dict['star']['type'] = 'phoenix'        #phoenix or user (if you have your own)
exo_dict['star']['mag'] = 7               #magnitude of the system
exo_dict['star']['ref_wave'] = 1.25         #For J mag = 1.25, H = 1.6, K =2.22.. etc (all in micron)
exo_dict['star']['temp'] = 5500             #in K 
exo_dict['star']['metal'] = 0.0             # as log Fe/H
exo_dict['star']['logg'] = 4.0              #log surface gravity cgs


exo_dict['planet']['type'] = 'constant'                  #tells pandexo you want a fixed transit depth
exo_dict['planet']['transit_duration'] = 2.0*60.0*60.0   #transit duration 
exo_dict['planet']['td_unit'] = 's' 
exo_dict['planet']['radius'] = 1
exo_dict['planet']['r_unit'] = 'R_jup'            #Any unit of distance in accordance with astropy.units can be added here
exo_dict['star']['radius'] = 1
exo_dict['star']['r_unit'] = 'R_sun'              #Same deal with astropy.units here
exo_dict['planet']['f_unit'] = 'rp^2/r*^2'        #this is what you would do for primary transit 


In [ ]:
print(jdi.subarrays('niriss'))

In [ ]:
inst_dict = jdi.load_mode_dict('NIRISS SOSS')

## Run all the NIRISS SOSS Subarrays and their computed group numbers

In [ ]:
all_results = {}
for isub in jdi.subarrays('niriss').keys(): 
    inst_dict["configuration"]["detector"]["subarray"]  = isub 
    inst_dict["configuration"]["detector"]["ngroup"]  = 'optimize' 
    result = jdi.run_pandexo(exo_dict, inst_dict,verbose=False)
    all_results[isub] = result


## Compare all the NIRISS SOSS Subarrays and their computed group numbers

In [ ]:
for isub in jdi.subarrays('niriss').keys(): 
    not_saturated = all_results[isub]['PandeiaOutTrans']['1d']['n_full_saturated'][1]
    not_isat = np.where(not_saturated==0)[0]
    x = all_results[isub]['FinalSpectrum']['wave'][not_isat]
    e = all_results[isub]['FinalSpectrum']['error_w_floor'][not_isat]
    grp_num = all_results[isub]['timing']['APT: Num Groups per Integration']
    plt.plot(x,e*1e6,label=isub + rf' | grp num={grp_num}')
plt.legend()
plt.xlabel('Wavelength')
plt.ylabel('PPM Precision')


In [ ]:
all_results['substrip256']['warning']
